## EDA

In [1]:
import numpy as np 
import pandas as pd

In [3]:
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')
print("train: ", train.shape, "test: ", test.shape)

train:  (90000, 14) test:  (30000, 13)


In [6]:
print(train.isna().sum())
train.head()

id                     0
customer_id            0
last_name              0
credit_score        9556
country             6021
gender                 0
age                    0
tenure                 0
acc_balance         7257
prod_count          4863
has_card               0
is_active              0
estimated_salary       0
exit_status            0
dtype: int64


,id,customer_id,last_name,credit_score,country,gender,age,tenure,acc_balance,prod_count,has_card,is_active,estimated_salary,exit_status
0,0,15788291,Iredale,559.0,France,Male,28.0,1,149989.39,1.0,1.0,1.0,67622.46,0
1,1,15642816,Hs?,694.0,France,Female,37.0,7,114510.35,2.0,0.0,0.0,182797.86,0
2,2,15632272,T'ien,585.0,NaN,Female,45.0,8,NaN,2.0,1.0,1.0,170338.35,0
3,3,15685826,Hightower,589.0,Spain,Male,25.0,0,166082.18,1.0,1.0,1.0,166476.46,0
4,4,15658032,Hopkins,701.0,France,Male,39.0,2,0.00,2.0,1.0,0.0,82526.92,0


In [7]:
print(test.isna().sum())
test.head()

id                     0
customer_id            0
last_name              0
credit_score        3185
country             4606
gender                 0
age                    0
tenure                 0
acc_balance         5251
prod_count          1717
has_card               0
is_active              0
estimated_salary       0
dtype: int64


,id,customer_id,last_name,credit_score,country,gender,age,tenure,acc_balance,prod_count,has_card,is_active,estimated_salary
0,0,15765283,T'ien,645.0,France,Male,30.0,10,0.00,2.0,0.0,0.0,85901.09
1,1,15660157,Macleod,641.0,France,Male,37.0,10,146573.68,3.0,1.0,0.0,168023.72
2,2,15621267,Ejimofor,637.0,France,Female,32.0,6,0.00,1.0,0.0,0.0,148769.08
3,3,15651280,Nnaife,714.0,France,Male,37.0,7,0.00,2.0,1.0,0.0,172576.22
4,4,15764294,Ifeatu,716.0,Germany,Male,31.0,4,98899.91,1.0,1.0,1.0,47832.82


In [6]:
import pandas as pd

# ============================================================
# Load datasets
# ============================================================

train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')

column_to_check = 'last_name'


# ============================================================
# 1. Basic information
# ============================================================

print("============================================================")
print(f"CHECKING OVERLAP FOR: {column_to_check}")
print("============================================================")

print(f"\nTrain rows: {len(train)}")
print(f"Test rows:  {len(test)}")

print(f"\nUnique {column_to_check} in train: {train[column_to_check].nunique()}")
print(f"Unique {column_to_check} in test:  {test[column_to_check].nunique()}")


# ============================================================
# 2. Unique values appearing in BOTH datasets
# ============================================================

train_values = set(train[column_to_check].dropna())
test_values = set(test[column_to_check].dropna())

matching_values = train_values & test_values

print("\n============================================================")
print("UNIQUE VALUE OVERLAP")
print("============================================================")

print(f"Unique matching {column_to_check}s: {len(matching_values)}")


# ============================================================
# 3. Total TEST ROWS whose value appears in TRAIN
# ============================================================

test_matching_mask = test[column_to_check].isin(train_values)

test_matching_rows = test_matching_mask.sum()

print("\n============================================================")
print("TEST ROW OCCURRENCE OVERLAP")
print("============================================================")

print(f"Test rows with {column_to_check} present in train: "
      f"{test_matching_rows}")

print(f"Percentage of test rows matching: "
      f"{test_matching_rows / len(test) * 100:.4f}%")


# ============================================================
# 4. Total TRAIN ROWS whose value appears in TEST
# ============================================================

train_matching_mask = train[column_to_check].isin(test_values)

train_matching_rows = train_matching_mask.sum()

print("\n============================================================")
print("TRAIN ROW OCCURRENCE OVERLAP")
print("============================================================")

print(f"Train rows with {column_to_check} present in test: "
      f"{train_matching_rows}")

print(f"Percentage of train rows matching: "
      f"{train_matching_rows / len(train) * 100:.4f}%")


# ============================================================
# 5. Frequency of matching values
# ============================================================

train_counts = train[column_to_check].value_counts()
test_counts = test[column_to_check].value_counts()

overlap_counts = pd.DataFrame({
    'train_count': train_counts,
    'test_count': test_counts
}).loc[list(matching_values)]

overlap_counts['total_count'] = (
    overlap_counts['train_count'] +
    overlap_counts['test_count']
)

overlap_counts = overlap_counts.sort_values(
    'total_count',
    ascending=False
)

print("\n============================================================")
print("TOP MATCHING VALUES")
print("============================================================")

print(overlap_counts.head(20))


# ============================================================
# 6. Exact row matches across all common columns
# ============================================================

common_columns = train.columns.intersection(test.columns)

train_common = train[common_columns].copy()
test_common = test[common_columns].copy()

exact_matches = test_common.merge(
    train_common.drop_duplicates(),
    how='inner',
    on=list(common_columns)
)

print("\n============================================================")
print("EXACT ROW MATCHES")
print("============================================================")

print(f"Common columns checked: {len(common_columns)}")
print(f"Exact matching test rows: {len(exact_matches)}")
print(f"Percentage of test exactly matching: "
      f"{len(exact_matches) / len(test) * 100:.4f}%")


# ============================================================
# 7. Summary
# ============================================================

print("\n============================================================")
print("SUMMARY")
print("============================================================")

print(f"""
Column checked:
    {column_to_check}

Unique values in train:
    {len(train_values):,}

Unique values in test:
    {len(test_values):,}

Unique values appearing in BOTH:
    {len(matching_values):,}

Test rows whose value appears in train:
    {test_matching_rows:,} / {len(test):,}
    ({test_matching_rows / len(test) * 100:.4f}%)

Train rows whose value appears in test:
    {train_matching_rows:,} / {len(train):,}
    ({train_matching_rows / len(train) * 100:.4f}%)

Exact matching test rows:
    {len(exact_matches):,} / {len(test):,}
    ({len(exact_matches) / len(test) * 100:.4f}%)
""")

CHECKING OVERLAP FOR: last_name

Train rows: 90000
Test rows:  30000

Unique last_name in train: 2611
Unique last_name in test:  2118

UNIQUE VALUE OVERLAP
Unique matching last_names: 2014

TEST ROW OCCURRENCE OVERLAP
Test rows with last_name present in train: 29867
Percentage of test rows matching: 99.5567%

TRAIN ROW OCCURRENCE OVERLAP
Train rows with last_name present in test: 88405
Percentage of train rows matching: 98.2278%

TOP MATCHING VALUES
                  train_count  test_count  total_count
last_name                                             
Hsia                   1343.0       457.0       1800.0
T'ien                  1245.0       408.0       1653.0
Kao                     896.0       279.0       1175.0
Hs?                     877.0       290.0       1167.0
Ts'ui                   845.0       319.0       1164.0
Maclean                 856.0       291.0       1147.0
P'eng                   823.0       277.0       1100.0
H?                      778.0       266.0       104